# Qwen2-VL 프롬프트 비교 테스트 v2

**v2 수정사항:** 프롬프트 단순화 + Few-shot 예시 + 반복방지 파라미터 + 프레임 이미지 동시 표시

**실행 전:** 런타임 > GPU (T4) 선택

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("GPU 확인 완료!")

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils
!pip install -q torch torchvision
print("설치 완료!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === 영상 경로 (필요시 수정) ===
VIDEO_PATH = "/content/drive/MyDrive/SampleVideo.Test/hyuck1.mp4"

import os
assert os.path.exists(VIDEO_PATH), f"파일 없음: {VIDEO_PATH}"
print(f"영상 확인: {VIDEO_PATH} ({os.path.getsize(VIDEO_PATH)/1024/1024:.1f}MB)")

In [ ]:
# 프레임 추출 (1fps) + 5장 균등 선택
import cv2
import numpy as np
from pathlib import Path

FRAME_DIR = Path("/content/test_frames")
FRAME_DIR.mkdir(exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
print(f"FPS: {fps:.1f} | 총 프레임: {total_frames} | 길이: {duration:.0f}초 ({duration/60:.1f}분)")

frame_interval = int(fps)
frame_paths = []
frame_idx = 0
saved = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        saved += 1
        path = FRAME_DIR / f"frame_{saved:06d}.jpg"
        cv2.imwrite(str(path), frame)
        frame_paths.append(path)
    frame_idx += 1
cap.release()

n_test = 5
indices = np.linspace(0, len(frame_paths)-1, n_test, dtype=int)
test_frames = [frame_paths[i] for i in indices]
print(f"추출: {len(frame_paths)}장 → 테스트: {n_test}장 선택")

In [ ]:
# 모델 로드
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("모델 로딩 중...")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("모델 로딩 완료!")

In [ ]:
# ── 프롬프트 정의 ──

# 기존 프롬프트 (AS-IS)
OLD_PROMPT = (
    "당신은 한국 TV 드라마 장면을 분석하여 광고 매칭에 활용할 컨텍스트를 생성하는 전문가입니다.\n\n"
    "(대사 없음)\n\n"
    "1~2문장으로 다음을 한국어로 설명하세요:\n"
    "- 등장인물과 그들이 하는 행동\n"
    "- 감정적 분위기\n"
    "- 인물들의 욕구나 필요\n\n"
    "사실에 근거하여 구체적으로 작성하세요. 광고 카테고리나 브랜드명은 언급하지 마세요.\n\n"
    "반드시 한국어로만 답하세요. Do not use English. 영어 사용 금지."
)

# 신규 프롬프트 v3 (TO-BE)
NEW_PROMPT = (
    "당신은 TV 드라마 장면을 분석하는 영상 분석 전문가입니다.\n"
    "화면에 보이는 것만 근거로 설명하세요. 추측하거나 지어내지 마세요.\n\n"
    "다음 3가지를 각각 1문장씩 한국어로 답하세요.\n\n"
    "1) 장소와 행동: 어떤 장소에서 누가 무엇을 하고 있는가?\n"
    "2) 분위기: 이 장면은 밝은가, 어두운가, 슬픈가, 즐거운가?\n"
    "3) 필요: 이 장면의 인물에게 필요한 것은 무엇인가?"
)


def run_old(frame_path):
    """기존 프롬프트"""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": str(frame_path)},
        {"type": "text", "text": OLD_PROMPT},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[str(frame_path)], padding=True, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=160)
    trimmed = generated[0][inputs['input_ids'].shape[-1]:]
    return processor.decode(trimmed, skip_special_tokens=True).strip()


def run_new(frame_path):
    """신규 프롬프트 v3 + 안정적 생성 파라미터"""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": str(frame_path)},
        {"type": "text", "text": NEW_PROMPT},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[str(frame_path)], padding=True, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=150,
            repetition_penalty=1.3,
            temperature=0.3,
            do_sample=True,
        )
    trimmed = generated[0][inputs['input_ids'].shape[-1]:]
    return processor.decode(trimmed, skip_special_tokens=True).strip()


print("준비 완료!")

In [ ]:
# ── 비교 테스트 실행 (프레임 이미지 + 결과 HTML 표시) ──
import time
from IPython.display import display, HTML
import base64

results = []

for i, frame_path in enumerate(test_frames):
    sec = indices[i]

    # 추론 실행
    t1 = time.time()
    old_result = run_old(frame_path)
    old_time = time.time() - t1

    t2 = time.time()
    new_result = run_new(frame_path)
    new_time = time.time() - t2

    results.append({
        "frame": frame_path.name, "sec": int(sec),
        "path": str(frame_path),
        "old": old_result, "new": new_result,
        "old_time": round(old_time, 1), "new_time": round(new_time, 1),
    })

    # 이미지를 base64로 인코딩
    with open(str(frame_path), "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    # 결과 텍스트 (300자 제한)
    old_display = old_result[:300] + "..." if len(old_result) > 300 else old_result
    new_display = new_result[:300] + "..." if len(new_result) > 300 else new_result

    html = f"""
    <div style="border:2px solid #ccc; border-radius:10px; padding:15px; margin:10px 0;">
      <h3 style="margin:0 0 10px 0;">{frame_path.name} (~{sec}s)</h3>
      <div style="display:flex; gap:15px; align-items:flex-start;">
        <div style="flex:1.2;">
          <img src="data:image/jpeg;base64,{img_b64}" style="width:100%; border-radius:8px;">
        </div>
        <div style="flex:1; background:#FFE0E0; border-radius:8px; padding:10px;">
          <b style="color:red;">[기존] ({old_time:.1f}s)</b>
          <p style="font-size:13px; margin-top:8px; white-space:pre-wrap;">{old_display}</p>
        </div>
        <div style="flex:1; background:#E0FFE0; border-radius:8px; padding:10px;">
          <b style="color:green;">[신규 v2] ({new_time:.1f}s)</b>
          <p style="font-size:13px; margin-top:8px; white-space:pre-wrap;">{new_display}</p>
        </div>
      </div>
    </div>
    """
    display(HTML(html))

print("모든 테스트 완료!")

In [ ]:
# ── 결과 요약표 ──
import pandas as pd

def check_format(text):
    return all(k in text for k in ["상황:", "감정:", "욕구:"])

def check_duplicate(text):
    words = text.split()
    if len(words) < 10:
        return False
    ngrams = [' '.join(words[i:i+5]) for i in range(len(words)-4)]
    return len(ngrams) != len(set(ngrams))

rows = []
for r in results:
    rows.append({
        "프레임": f"{r['frame']} (~{r['sec']}s)",
        "기존-중복": "X" if check_duplicate(r['old']) else "OK",
        "기존-글자수": len(r['old']),
        "신규-형식": "OK" if check_format(r['new']) else "X",
        "신규-중복": "X" if check_duplicate(r['new']) else "OK",
        "신규-글자수": len(r['new']),
    })

df = pd.DataFrame(rows)
display(df)

fmt = sum(1 for r in results if check_format(r['new'])) / len(results) * 100
old_dup = sum(1 for r in results if check_duplicate(r['old'])) / len(results) * 100
new_dup = sum(1 for r in results if check_duplicate(r['new'])) / len(results) * 100

print(f"\n신규 형식 준수율: {fmt:.0f}%")
print(f"기존 중복률: {old_dup:.0f}% -> 신규 중복률: {new_dup:.0f}%")